In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import matplotlib.pyplot as plt
import hierarchicalforecast
from hierarchicalforecast.utils import aggregate
df1 = pd.read_csv("2012-2013 Solar home electricity data v2.csv", low_memory=False)
df1.columns = df1.iloc[0,:].values
df1 = df1.iloc[1:, :-1].reset_index(drop = True) # all values of column "row quality" are non
df = df1[df1['Consumption Category'] == 'GC']
b = [len(df[df['Customer'] == f"{i}"]) == 365 for i in tqdm(range(1, 301))] #check length
df = pd.concat([df[df['Customer'] == f"{int(i)+1}"] for i in tqdm(np.where(np.array(b))[0])])

100%|███████████████████████████████████████████████████████████████████████████████| 299/299 [00:02<00:00, 134.82it/s]


In [2]:
df

,Customer,Generator Capacity,Postcode,Consumption Category,date,0:30,1:00,1:30,2:00,2:30,...,19:30,20:00,20:30,21:00,21:30,22:00,22:30,23:00,23:30,0:00
1,1,3.78,2076,GC,1/07/2012,0.855,0.786,0.604,0.544,0.597,...,0.329,0.374,0.447,0.549,0.136,0.288,0.181,0.651,0.09,0.068
4,1,3.78,2076,GC,2/07/2012,0.309,0.082,0.059,0.097,0.29,...,0.696,0.353,0.464,0.229,0.811,0.222,0.306,1.034,0.136,0.067
7,1,3.78,2076,GC,3/07/2012,0.092,0.076,0.318,0.088,0.061,...,0.268,0.33,0.654,0.406,0.141,0.073,0.19,0.902,0.098,0.066
10,1,3.78,2076,GC,4/07/2012,0.081,0.082,0.306,0.098,0.725,...,0.635,0.403,0.204,0.286,0.203,0.521,0.259,1.306,0.259,0.26
13,1,3.78,2076,GC,5/07/2012,0.445,0.255,0.138,0.115,0.071,...,0.208,0.24,0.506,0.237,0.257,0.462,0.414,0.932,0.07,0.094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268543,300,3.36,2086,GC,26/06/2013,0.124,0.157,0.131,0.15,0.15,...,0.51,0.614,0.458,0.535,0.427,0.315,0.253,0.43,0.881,0.853
268546,300,3.36,2086,GC,27/06/2013,0.25,0.153,0.191,0.152,0.176,...,0.654,0.696,0.662,0.584,0.682,0.666,0.63,0.417,0.295,0.607
268549,300,3.36,2086,GC,28/06/2013,0.548,0.817,0.155,0.187,0.171,...,1.564,1.319,1.259,0.921,0.667,0.495,0.486,0.389,0.448,0.269
268552,300,3.36,2086,GC,29/06/2013,0.171,0.832,0.44,0.745,0.149,...,0.436,0.383,0.404,0.367,0.398,0.35,0.228,0.17,0.139,0.171


In [3]:
id_vars = ['Customer', 'Generator Capacity', 'Postcode', 'Consumption Category', 'date']
time_cols = [col for col in df.columns if col not in id_vars]
df_long = df.melt(
    id_vars=id_vars,
    value_vars=time_cols,
    var_name='time',
    value_name='consumption'
)
# df_long = df_long[df_long['Consumption Category'] == 'GC']
a = df_long['date']+' '+df_long['time']
a = pd.to_datetime(a, format='%d/%m/%Y %H:%M')
df_long['ds'] = a
dff = df_long.drop(columns = ['date','time', 'Generator Capacity']).copy()
del df_long
dff['Customer'] = dff['Customer'].astype('int') #.sort_values()
dff = dff.sort_values(by=['Customer','ds'], ascending=True)
dff = dff.reset_index(drop = True)

In [4]:
dff.columns = ["Customer", "Postcode","Consumption Category", "y", "ds"]
dff['y'] = dff['y'].astype('float')
SPEC = [
    ["Consumption Category"],  # Level 1
    ["Consumption Category", "Postcode"],  # Level 2
    ["Consumption Category", "Postcode", "Customer"]
]
Y_df, S_df, tags = aggregate(dff, SPEC)
#Y_df['levels'] = Y_df['unique_id'].str.count('/') + 1

In [5]:
arr = Y_df['unique_id'].unique()

label2uid = {label: i for i, label in enumerate(arr)}
uid2label = {i: label for label, i in label2uid.items()} 

codes, uniques = pd.factorize(arr)   # codes: 每个元素对应的整数编码
label2id = dict(zip(uniques, range(len(uniques))))

Y_df['partition_id'] = Y_df['unique_id'].map(label2id)

#Y_df = Y_df[Y_df['ds'] >= '2017-04-01']

Y_df

,unique_id,ds,y,partition_id
0,GC,2012-07-01 00:00:00,102.719,0
1,GC,2012-07-01 00:30:00,83.000,0
2,GC,2012-07-01 01:00:00,75.919,0
3,GC,2012-07-01 01:30:00,75.508,0
4,GC,2012-07-01 02:00:00,65.438,0
...,...,...,...,...
7007995,GC/2330/85,2013-06-30 21:30:00,1.165,399
7007996,GC/2330/85,2013-06-30 22:00:00,0.307,399
7007997,GC/2330/85,2013-06-30 22:30:00,0.305,399
7007998,GC/2330/85,2013-06-30 23:00:00,0.316,399


In [6]:
Y_df.to_csv("AusGrid2013_univariate.csv", index = False)
S_df.to_csv('AusGrid_S.csv', index = False)
import pickle
with open("tags.pkl", "wb") as f:
    pickle.dump(tags, f)

# Check

In [14]:
import pandas as pd
import numpy as np
from tqdm import tqdm 
import matplotlib.pyplot as plt
import hierarchicalforecast
from hierarchicalforecast.utils import aggregate
df = pd.read_csv("2012-2013 Solar home electricity data v2.csv", low_memory=False)
df.columns = df.iloc[0,:].values
df = df.iloc[1:, :-1].reset_index(drop = True) # all values of column "row quality" are non
id_vars = ['Customer', 'Generator Capacity', 'Postcode', 'Consumption Category', 'date']
time_cols = [col for col in df.columns if col not in id_vars]
df_long = df.melt(
    id_vars=id_vars,
    value_vars=time_cols,
    var_name='time',
    value_name='consumption'
)
df_long = df_long[df_long['Consumption Category'] == 'GC']
a = df_long['date']+' '+df_long['time']
a = pd.to_datetime(a, format='%d/%m/%Y %H:%M')
df_long['ds'] = a
dff = df_long.drop(columns = ['date','time', 'Generator Capacity']).copy()
#del df_long
dff['Customer'] = dff['Customer'].astype('int') #.sort_values()
dff = dff.sort_values(by=['Customer','ds'], ascending=True)
dff = dff.reset_index(drop = True)

In [17]:
a = df[df['Consumption Category'] == 'GC']

In [27]:

b

100%|███████████████████████████████████████████████████████████████████████████████| 300/300 [00:02<00:00, 131.13it/s]


[True,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True

array([  0,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

In [39]:
np.array(b)

array([ True, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,